In [ ]:
pip install mne numpy pandas scipy pywt antropy
# This script installs the necessary Python packages for EEG signal processing and analysis.


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 2.4 MB/s  0:00:03 eta 0:00:01
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/67/64/4cb909dd5ab09a9a5d086eff9586e69e827b88a5585517386879474f4cf7/numpy-2.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.4 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/e5/63/cd7d615331b328e287d8233ba9fdf191a9c2d11b6af0c7a59cfcec23de68/pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.4 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/3e/f3/c5195b1ae57ef85339fd7285dfb603b22c8b4e79114bae5f4f0fcf688677/matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.7 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/79/2

In [2]:
import mne
import numpy as np
from pathlib import Path


In [3]:
EPOCH_LENGTH = 30        # seconds
CHANNEL_NAME = 'EEG Fpz-Cz'  # atau 'EEG Pz-Oz'


In [4]:
def load_eeg(edf_path, channel_name):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    raw.pick_channels([channel_name])
    fs = int(raw.info['sfreq'])
    signal = raw.get_data()[0]  # shape: (n_samples,)
    return signal, fs


In [5]:
def load_hypnogram(hyp_path):
    ann = mne.read_annotations(hyp_path)
    return ann


In [6]:
SLEEP_STAGE_MAP = {
    'Sleep stage W': 0,
    'Sleep stage N1': 1,
    'Sleep stage N2': 2,
    'Sleep stage N3': 3,
    'Sleep stage R': 4
}


In [7]:
def epoch_signal(signal, fs, epoch_length):
    samples_per_epoch = fs * epoch_length
    n_epochs = len(signal) // samples_per_epoch
    epochs = np.array([
        signal[i*samples_per_epoch:(i+1)*samples_per_epoch]
        for i in range(n_epochs)
    ])
    return epochs


In [8]:
def extract_labels(annotations, n_epochs):
    labels = []
    for ann in annotations:
        if ann['description'] in SLEEP_STAGE_MAP:
            labels.append(SLEEP_STAGE_MAP[ann['description']])
    
    labels = np.array(labels)
    return labels[:n_epochs]


In [9]:
def process_subject(eeg_path, hyp_path, subject_id):
    signal, fs = load_eeg(eeg_path, CHANNEL_NAME)
    annotations = load_hypnogram(hyp_path)
    
    X = epoch_signal(signal, fs, EPOCH_LENGTH)
    y = extract_labels(annotations, len(X))
    
    subjects = np.array([subject_id] * len(y))
    
    return X, y, subjects, fs


In [10]:
X_all, y_all, subj_all = [], [], []

for eeg_file in Path("sleep-edf").glob("*PSG.edf"):
    hyp_file = eeg_file.with_name(eeg_file.name.replace("PSG", "Hypnogram"))
    subject_id = eeg_file.stem.split('-')[0]
    
    X, y, s, fs = process_subject(eeg_file, hyp_file, subject_id)
    
    X_all.append(X)
    y_all.append(y)
    subj_all.append(s)

X_all = np.concatenate(X_all)
y_all = np.concatenate(y_all)
subj_all = np.concatenate(subj_all)


ValueError: need at least one array to concatenate

In [ ]:
from scipy.stats import skew, kurtosis
import numpy as np

def time_features(epoch):
    return {
        'mean': np.mean(epoch),
        'std': np.std(epoch),
        'var': np.var(epoch),
        'rms': np.sqrt(np.mean(epoch**2)),
        'skew': skew(epoch),
        'kurtosis': kurtosis(epoch),
        'ptp': np.ptp(epoch),
        'zcr': np.mean(np.diff(np.sign(epoch)) != 0)
    }


In [ ]:
def hjorth_features(epoch):
    diff1 = np.diff(epoch)
    diff2 = np.diff(diff1)
    
    var0 = np.var(epoch)
    var1 = np.var(diff1)
    var2 = np.var(diff2)
    
    activity = var0
    mobility = np.sqrt(var1 / var0)
    complexity = np.sqrt(var2 / var1) / mobility
    
    return {
        'hjorth_activity': activity,
        'hjorth_mobility': mobility,
        'hjorth_complexity': complexity
    }


In [ ]:
from scipy.signal import welch

BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'sigma': (12, 16),
    'beta': (16, 30)
}

def bandpower(freqs, psd, band):
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return np.trapz(psd[idx], freqs[idx])

def freq_features(epoch, fs):
    freqs, psd = welch(epoch, fs=fs, nperseg=fs*2)
    total_power = np.trapz(psd, freqs)
    
    feats = {'total_power': total_power}
    
    for name, band in BANDS.items():
        bp = bandpower(freqs, psd, band)
        feats[f'{name}_abs_power'] = bp
        feats[f'{name}_rel_power'] = bp / total_power
    
    return feats


In [ ]:
import antropy as ant

def entropy_features(epoch):
    return {
        'perm_entropy': ant.perm_entropy(epoch, normalize=True),
        'sample_entropy': ant.sample_entropy(epoch),
        'spectral_entropy': ant.spectral_entropy(epoch, sf=100),
        'lz_complexity': ant.lziv_complexity(epoch, normalize=True),
        'dfa': ant.detrended_fluctuation(epoch)
    }


In [ ]:
import pywt

def wavelet_features(epoch, wavelet='db4', level=5):
    coeffs = pywt.wavedec(epoch, wavelet, level=level)
    feats = {}
    
    for i, c in enumerate(coeffs):
        feats[f'wav_L{i}_mean'] = np.mean(c)
        feats[f'wav_L{i}_std'] = np.std(c)
        feats[f'wav_L{i}_energy'] = np.sum(c**2)
        feats[f'wav_L{i}_entropy'] = ant.perm_entropy(c, normalize=True)
    
    return feats


In [ ]:
def extract_features(epoch, fs):
    features = {}
    features.update(time_features(epoch))
    features.update(hjorth_features(epoch))
    features.update(freq_features(epoch, fs))
    features.update(entropy_features(epoch))
    features.update(wavelet_features(epoch))
    return features


In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score
)


In [ ]:
X_features  # pandas DataFrame (n_epochs × n_features)
y_labels    # numpy array (n_epochs,)
subjects    # numpy array (n_epochs,)


In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)


In [ ]:
metrics = {
    'f1_macro': [],
    'accuracy': [],
    'balanced_accuracy': [],
    'kappa': []
}

for fold, (train_idx, test_idx) in enumerate(
        sgkf.split(X_features, y_labels, groups=subjects), 1):

    X_train = X_features.iloc[train_idx]
    X_test  = X_features.iloc[test_idx]
    y_train = y_labels[train_idx]
    y_test  = y_labels[test_idx]

    # Scaling (fit ONLY on train)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Train
    rf_model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = rf_model.predict(X_test_scaled)

    # Metrics
    metrics['f1_macro'].append(
        f1_score(y_test, y_pred, average='macro')
    )
    metrics['accuracy'].append(
        accuracy_score(y_test, y_pred)
    )
    metrics['balanced_accuracy'].append(
        balanced_accuracy_score(y_test, y_pred)
    )
    metrics['kappa'].append(
        cohen_kappa_score(y_test, y_pred)
    )

    print(f"Fold {fold} selesai.")


In [ ]:
baseline_results = {
    metric: (np.mean(scores), np.std(scores))
    for metric, scores in metrics.items()
}

baseline_df = pd.DataFrame(
    baseline_results,
    index=['Mean ± Std']
).T

baseline_df


In [ ]:
import numpy as np
import pandas as pd

import shap
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score
)


In [ ]:
N_SPLITS = 5
TOP_K_RATIO = 0.55   # 55% fitur teratas (aman & premium)
RANDOM_STATE = 42


In [ ]:
def build_xgb():
    return XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


In [ ]:
results = {
    'xgb_all': {'f1': [], 'acc': [], 'bal_acc': [], 'kappa': []},
    'xgb_shap': {'f1': [], 'acc': [], 'bal_acc': [], 'kappa': []}
}


In [ ]:
for fold, (train_idx, test_idx) in enumerate(
        sgkf.split(X_features, y_labels, groups=subjects), 1):

    print(f"\nFold {fold}")

    # Split
    X_train = X_features.iloc[train_idx]
    X_test  = X_features.iloc[test_idx]
    y_train = y_labels[train_idx]
    y_test  = y_labels[test_idx]

    # Scaling (FIT ONLY ON TRAIN)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # ===============================
    # 1) XGBoost - ALL FEATURES
    # ===============================
    xgb_all = build_xgb()
    xgb_all.fit(X_train_s, y_train)

    y_pred_all = xgb_all.predict(X_test_s)

    results['xgb_all']['f1'].append(
        f1_score(y_test, y_pred_all, average='macro')
    )
    results['xgb_all']['acc'].append(
        accuracy_score(y_test, y_pred_all)
    )
    results['xgb_all']['bal_acc'].append(
        balanced_accuracy_score(y_test, y_pred_all)
    )
    results['xgb_all']['kappa'].append(
        cohen_kappa_score(y_test, y_pred_all)
    )

    # ===============================
    # 2) SHAP FEATURE SELECTION
    # ===============================
    explainer = shap.TreeExplainer(xgb_all)
    shap_values = explainer.shap_values(X_train_s)

    # Multi-class -> average |SHAP| over class & sample
    shap_importance = np.mean(
        np.abs(np.array(shap_values)),
        axis=(0, 1)
    )

    shap_df = pd.DataFrame({
        'feature': X_features.columns,
        'importance': shap_importance
    }).sort_values('importance', ascending=False)

    TOP_K = int(len(shap_df) * TOP_K_RATIO)
    selected_features = shap_df.head(TOP_K)['feature'].tolist()

    # ===============================
    # 3) XGBoost - SHAP FEATURES
    # ===============================
    X_train_sel = scaler.fit_transform(X_train[selected_features])
    X_test_sel  = scaler.transform(X_test[selected_features])

    xgb_shap = build_xgb()
    xgb_shap.fit(X_train_sel, y_train)

    y_pred_shap = xgb_shap.predict(X_test_sel)

    results['xgb_shap']['f1'].append(
        f1_score(y_test, y_pred_shap, average='macro')
    )
    results['xgb_shap']['acc'].append(
        accuracy_score(y_test, y_pred_shap)
    )
    results['xgb_shap']['bal_acc'].append(
        balanced_accuracy_score(y_test, y_pred_shap)
    )
    results['xgb_shap']['kappa'].append(
        cohen_kappa_score(y_test, y_pred_shap)
    )


In [ ]:
def summarize(results_dict):
    summary = {}
    for model, metrics in results_dict.items():
        summary[model] = {
            m: (np.mean(v), np.std(v))
            for m, v in metrics.items()
        }
    return pd.DataFrame(summary)

summary_df = summarize(results)
summary_df


In [ ]:
from scipy.stats import wilcoxon

stat, p_value = wilcoxon(
    results['xgb_all']['f1'],
    results['xgb_shap']['f1']
)

print(f"Wilcoxon p-value (Macro F1): {p_value:.4f}")
